In [0]:
%pip install sentence-transformers

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [0]:
def retrieve_candidates(query, k=20):
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    return results["documents"][0], results["metadatas"][0]

In [0]:
def rerank_with_cross_encoder(query, docs, metadata, top_k=5):
    pairs = [(query, doc) for doc in docs]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs, metadata),
        key=lambda x: x[0],
        reverse=True
    )

    return ranked[:top_k]

In [0]:
LEGAL_KEYWORDS = {
    "penalty": ["penalty", "fine", "punishable", "imprisonment"],
    "obligation": ["shall", "must", "required"],
    "rights": ["right", "entitled", "freedom"],
    "exception": ["provided that", "exception"],
    "procedure": ["procedure", "process", "steps"]
}

def classify_legal_context(text):
    categories = []

    lower = text.lower()

    for label, words in LEGAL_KEYWORDS.items():
        if any(w in lower for w in words):
            categories.append(label)

    return categories

In [0]:
def enhance_context(ranked):
    enhanced = []

    for score, doc, meta in ranked:
        tags = classify_legal_context(doc)

        enhanced.append({
            "text": doc,
            "section": meta.get("section", ""),
            "tags": tags,
            "score": score
        })

    return enhanced

In [0]:
def extract_citations(enhanced):
    citations = set()

    for item in enhanced:
        if item["section"]:
            citations.add(item["section"])

    return citations

In [0]:
def build_grounded_context(enhanced):
    context = []

    for item in enhanced:
        if len(item["text"]) > 120:
            context.append(item["text"])

    return context[:3]

In [0]:
def build_expert_prompt(query, context):

    context_text = "\n\n".join(context)

    return f"""
You are an expert Indian legal assistant.

Use ONLY the legal context provided.

Provide accurate and clear legal explanation.

Include:
• relevant law
• penalties & consequences
• citizen responsibilities
• exceptions if applicable

Question:
{query}

Legal Context:
{context_text}

Provide a structured answer.
"""

In [0]:
def generate_expert_answer(query):

    docs, metadata = retrieve_candidates(query)

    ranked = rerank_with_cross_encoder(query, docs, metadata)

    enhanced = enhance_context(ranked)

    context = build_grounded_context(enhanced)

    citations = extract_citations(enhanced)

    prompt = build_expert_prompt(query, context)

    response = llm(prompt)[0]["generated_text"]

    return response, citations

In [0]:
def format_expert_output(answer, citations):
    return f"""
⚖️ LEGAL EXPLANATION

{answer}

📚 Relevant Sections:
{", ".join(citations) if citations else "Applicable provisions"}

🧾 Practical Guidance:
Follow safety and legal requirements to avoid penalties.

⚠️ DISCLAIMER:
This AI-generated response provides legal information and is not a substitute for professional legal advice.
"""

In [0]:
def ask_expert_legal_assistant(query):
    answer, citations = generate_expert_answer(query)
    return format_expert_output(answer, citations)

In [0]:
print(ask_expert_legal_assistant(
    "Penalty for not wearing helmet in India"
))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7985450412821853>, line 1
----> 1 print(ask_expert_legal_assistant(
      2     "Penalty for not wearing helmet in India"
      3 ))

File <command-7985450412821852>, line 2, in ask_expert_legal_assistant(query)
      1 def ask_expert_legal_assistant(query):
----> 2     answer, citations = generate_expert_answer(query)
      3     return format_expert_output(answer, citations)

File <command-7985450412821850>, line 3, in generate_expert_answer(query)
      1 def generate_expert_answer(query):
----> 3     docs, metadata = retrieve_candidates(query)
      5     ranked = rerank_with_cross_encoder(query, docs, metadata)
      7     enhanced = enhance_context(ranked)

File <command-7985450412821823>, line 2, in retrieve_candidates(query, k)
      1 def retrieve_candidates(query, k=20):
----> 2     results = collection.query(
  